# Projet 8 — Analysez l’évolution des prix de l’immobilier avec Python
**OpenClassrooms · Business Intelligence Analyst · Steve Bell**

## Les Plus Beaux Logis de Paris

L’objectif est d’analyser les ventes parisiennes de 2017 à 2021, d’entraîner une
régression linéaire pour estimer la **valeur foncière totale**, de valoriser le
portefeuille au **31 décembre 2022**, puis de proposer un classement des biens
de Louise avec **K-means**. Les cinq étapes du notebook fourni sont conservées.

Les résultats décrivent les fichiers pédagogiques du projet. La valorisation
2022 est une extrapolation statistique, et le classement de Louise reste une
hypothèse tant que les types réels des biens ne sont pas disponibles.

## Synthèse de l’exécution vérifiée

1. **Données :** 26 196 lignes de ventes du 02/01/2017 au 31/12/2021,
   dont 24 353 appartements et 1 843 locaux.
2. **Prix des appartements :** 10,1 % entre les moyennes 2017 et 2021 ;
   -2,1 % de 2020 à 2021. Le prix moyen varie aussi selon l’arrondissement.
3. **Régression :** MAPE de 9,14 % sur le test de 33 %,
   et de 13,62 % sur le test chronologique 2021.
   Ce dernier présente un biais relatif moyen de 12,81 %.
4. **Portefeuille au 31/12/2022 :** 169,70 M€ estimés pour
   275 actifs, dont 71,22 M€
   pour les particuliers et 98,48 M€
   pour le corporate, selon la convention fondée sur le type de bien. Le total corporate est supérieur.
5. **Louise :** 40 biens séparés en deux groupes de
   20 et 20 observations.
   Silhouette : 0,909. Les types proposés ne sont pas validés par des labels réels.

*Résumé de cette version, vérifiée le 16 novembre 2026. La synthèse de la section 5.1 est recalculée à chaque exécution.*

## Plan et mode d’emploi

1. [Analyse des données](#milestone-1) : qualité, types de biens, volumes, période,
   prix au m², arrondissements et corrélations.
2. [Entraînement de l’algorithme](#milestone-2) : régression linéaire, test de 33 %,
   mesures d’erreur et contrôle chronologique.
3. [Prédiction pour le client](#milestone-3) : portefeuille au 31/12/2022 et comparaison des segments.
4. [Classification des données](#milestone-4) : K-means sur l’échantillon, interprétation et limites.
5. [Analyse et présentation des résultats](#milestone-5) : conclusions, recommandations et livrables.

**Exécution :** sélectionner un noyau Python 3.10 ou supérieur, puis
« Redémarrer le noyau et exécuter toutes les cellules ». Le notebook fonctionne
depuis la racine du projet, son dossier `livrables`, ou le dossier du ZIP fourni.
Les trois classeurs doivent rester dans `data/raw/`. Le paramètre `DOSSIER_DONNEES`
permet aussi d’indiquer leur emplacement explicitement.

Installation, dans un terminal de l’environnement Python utilisé :

```bash
python -m pip install pandas numpy matplotlib scipy scikit-learn openpyxl ipykernel nbformat nbclient nbconvert
```

Les résultats sont intégrés au notebook et les exports sont écrits dans
`livrables/`. Aucune connexion Internet n’est nécessaire pour recalculer le projet.

## Sources et hypothèses

| Fichier | Rôle | Informations utilisées |
|---|---|---|
| `historique_immobilier_paris_2017_2021.xlsx` | Analyse et apprentissage | Date, valeur foncière, surface réelle, code postal et type de bien |
| `portefeuille_actifs.xlsx` | Estimation au 31/12/2022 | Surface réelle bâtie, localisation et type de bien |
| `echantillon_a_classer2.xlsx` | Classement des opportunités de Louise | Valeur foncière et surface réelle ; localisation pour le contexte |

Les fichiers proviennent du [dépôt P8 de Steve Bell](https://github.com/steve57000/p8/tree/b119753a763b474d1fa116897f449944be255b49/data/raw),
version consultée pour cette révision. L’échantillon est également proposé dans
les [ressources OpenClassrooms](https://s3.eu-west-1.amazonaws.com/course.oc-static.com/projects/Business+Intelligence+Analyst/BIA_project+8/echantillon_a_classer2.xlsx).
Les empreintes des fichiers utilisés sont enregistrées à la fin du notebook.

**Conventions importantes :**

- Une ligne historique représente une transaction dans cet exercice. Sans identifiant de mutation,
  le nombre exact de ventes distinctes ne peut pas être garanti.
- Le prix au m² est `valeur_fonciere / surface_reelle`. La moyenne présentée donne
  le même poids à chaque ligne ; ce n’est pas le ratio des sommes des prix et des surfaces.
- Les dates, surfaces et prix doivent être valides ; les anomalies bloquantes sont signalées,
  sans correction ni suppression silencieuse.
- Les deux segments sont définis par convention : appartements → particuliers ;
  locaux industriels, commerciaux ou assimilés → corporate. Le fichier ne fournit pas de branche commerciale.
- L’année 2021 de l’échantillon de Louise est une indication du sujet : le classeur ne contient
  aucune date permettant de la vérifier. Les deux classes demandées ne permettent pas
  de distinguer commerce, cabinet et agence au sein des locaux.

## Préparation — bibliothèques, paramètres et affichage

In [ ]:
from pathlib import Path
from hashlib import sha256
from importlib.metadata import version
import json
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.dates as mdates
from IPython.display import display, Markdown
from scipy.stats import pearsonr, spearmanr
from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error, mean_absolute_percentage_error, mean_squared_error,
    r2_score, silhouette_score, accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

GRAINE = 42
PART_TEST = 0.33
DATE_ORIGINE = pd.Timestamp('2017-01-01')
DATE_ESTIMATION = pd.Timestamp('2022-12-31')
ANNEE_ECHANTILLON = 2021  # Indication du sujet, non vérifiable dans le classeur.
DOSSIER_DONNEES = "../data/raw"  # Exemple Windows : Path('C:/projets/p8/data/raw')

LIBELLES_TYPES = {2: 'Appartement', 4: 'Local industriel. commercial ou assimilé'}
SEGMENTS = {2: 'Particuliers', 4: 'Corporate'}
BLEU, ORANGE, GRIS = '#245C8A', '#BC6C25', '#56616E'

plt.rcParams.update({
    'figure.figsize': (10, 4.5), 'figure.dpi': 110, 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11, 'axes.spines.top': False,
    'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': 0.18,
    'axes.axisbelow': True, 'figure.constrained_layout.use': True,
})
pd.set_option('display.max_rows', 25)
pd.set_option('display.max_columns', 12)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' ').replace('.', ','))

def fr(nombre, decimales=0):
    return f'{nombre:,.{decimales}f}'.replace(',', ' ').replace('.', ',')

def afficher_p(p):
    # Une p-value arrondie à zéro n'est pas une probabilité strictement nulle.
    return '< 10⁻¹⁶' if p < 1e-16 else f'{p:.3g}'

def terminer_figure(fig, nom):
    fig.savefig(DOSSIER_FIGURES / f'{nom}.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)

format_euros = mtick.FuncFormatter(lambda x, pos: fr(x))

### Localiser les fichiers et préparer les exports

In [ ]:
FICHIERS = {
    'historique': 'historique_immobilier_paris_2017_2021.xlsx',
    'portefeuille': 'portefeuille_actifs.xlsx',
    'echantillon': 'echantillon_a_classer2.xlsx',
}

candidats = [Path(DOSSIER_DONNEES)] if DOSSIER_DONNEES is not None else [
    Path.cwd() / 'data' / 'raw',
    Path.cwd().parent / 'data' / 'raw',
    Path.cwd(),
]
DATA_DIR = next(
    (d.resolve() for d in candidats if all((d / nom).is_file() for nom in FICHIERS.values())),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError(
        'Les trois classeurs sont nécessaires. Placez-les dans data/raw près du notebook '
        'ou renseignez DOSSIER_DONNEES dans la cellule de paramètres.'
    )
RACINE = DATA_DIR.parent.parent if DATA_DIR.name == 'raw' and DATA_DIR.parent.name == 'data' else DATA_DIR
DOSSIER_RESULTATS = RACINE / 'livrables'
DOSSIER_FIGURES = DOSSIER_RESULTATS / 'figures'
DOSSIER_FIGURES.mkdir(parents=True, exist_ok=True)
print('Trois fichiers sources trouvés. Les exports seront placés dans livrables/.')

def verifier_colonnes(df, colonnes, nom):
    manquantes = sorted(set(colonnes) - set(df.columns))
    if manquantes:
        raise ValueError(f'{nom} : colonnes manquantes {manquantes}')

<a id="milestone-1"></a>
## Milestone 1 — Analyse des données

### 1.1. Importer les données et vérifier les types

Je conserve une copie brute avant toute transformation. Les codes postaux et
codes de type sont des catégories, même lorsqu’Excel les stocke comme des nombres.

In [ ]:
historique_brut = pd.read_excel(DATA_DIR / FICHIERS['historique'], engine='openpyxl')
verifier_colonnes(historique_brut, [
    'date_mutation', 'valeur_fonciere', 'surface_reelle',
    'code_postal', 'code_type_local', 'type_local',
], 'Historique')
historique = historique_brut.copy()
historique['date_mutation'] = pd.to_datetime(historique['date_mutation'], errors='coerce')
for colonne in ['valeur_fonciere', 'surface_reelle', 'code_postal', 'code_type_local']:
    historique[colonne] = pd.to_numeric(historique[colonne], errors='coerce')

profil = pd.DataFrame({
    'Type initial': historique_brut.dtypes.astype(str),
    'Type après conversion': historique.dtypes.astype(str),
    'Valeurs manquantes': historique.isna().sum(),
})
display(profil)
display(historique.head(5))

### Contrôler la qualité et les lignes identiques

Les contrôles portent sur les valeurs manquantes, les prix et surfaces positifs,
la période attendue, les 20 arrondissements et la correspondance code/libellé.
Une valeur élevée n’est pas supprimée uniquement parce qu’elle est inhabituelle.

In [ ]:
colonnes_numeriques = ['valeur_fonciere', 'surface_reelle', 'code_postal', 'code_type_local']
controles_historique = pd.Series({
    'Valeurs manquantes après conversion': int(historique.isna().sum().sum()),
    'Valeurs numériques non finies': int((~np.isfinite(historique[colonnes_numeriques])).sum().sum()),
    'Prix nuls ou négatifs': int(historique['valeur_fonciere'].le(0).sum()),
    'Surfaces nulles ou négatives': int(historique['surface_reelle'].le(0).sum()),
    'Dates hors 2017–2021': int((~historique['date_mutation'].between('2017-01-01', '2021-12-31')).sum()),
    'Codes postaux hors Paris 1 à 20': int((~historique['code_postal'].isin(range(75001, 75021))).sum()),
    'Codes de type non attendus': int((~historique['code_type_local'].isin(LIBELLES_TYPES)).sum()),
    'Incohérences code / libellé': int(historique['type_local'].ne(historique['code_type_local'].map(LIBELLES_TYPES)).sum()),
}, name='Nombre d’anomalies')
display(controles_historique.to_frame())
if controles_historique.gt(0).any():
    raise ValueError('Le fichier historique contient des anomalies à examiner avant de poursuivre.')

nb_identiques_historique = int(historique_brut.duplicated().sum())
historique['code_postal'] = historique['code_postal'].astype(int).astype(str)
historique['code_type_local'] = historique['code_type_local'].astype(int)
display(Markdown(
    f'**Bilan :** {fr(len(historique))} lignes, aucune anomalie bloquante et '
    f'**{nb_identiques_historique} répétitions exactes** au-delà de la première occurrence. '
    'Elles sont conservées dans les statistiques descriptives faute d’identifiant de mutation. '
    'Pour le modèle, une seule occurrence sera retenue afin d’éviter de placer une même '
    'observation dans l’apprentissage et dans le test.'
))

### 1.2. Identifier les types de biens

In [ ]:
types_biens = (
    historique.groupby(['code_type_local', 'type_local']).size()
    .rename('Nombre de lignes').reset_index()
)
types_biens['Part (%)'] = 100 * types_biens['Nombre de lignes'] / len(historique)
display(types_biens)
display(Markdown(
    f'Le fichier contient **{len(types_biens)} types de biens** : '
    '**Appartement (code 2)** et **Local industriel, commercial ou assimilé (code 4)**. '
    'La seconde catégorie regroupe plusieurs usages ; elle ne décrit pas uniquement des commerces.'
))

### 1.3. Compter les transactions représentées

In [ ]:
nb_appartements = int(historique['code_type_local'].eq(2).sum())
nb_locaux = int(historique['code_type_local'].eq(4).sum())
assert nb_appartements + nb_locaux == len(historique)
display(pd.DataFrame({
    'Population': ['Total', 'Appartements', 'Locaux industriels, commerciaux ou assimilés'],
    'Nombre de lignes / transactions représentées': [len(historique), nb_appartements, nb_locaux],
}))

### 1.4. Vérifier l’intervalle historique et calculer le prix au m²

In [ ]:
date_min = historique['date_mutation'].min()
date_max = historique['date_mutation'].max()
historique['annee'] = historique['date_mutation'].dt.year
historique['date_annees'] = (historique['date_mutation'] - DATE_ORIGINE).dt.days / 365.25
historique['prix_m2'] = historique['valeur_fonciere'] / historique['surface_reelle']
assert np.isfinite(historique['prix_m2']).all()
print(f'Historique disponible : du {date_min:%d/%m/%Y} au {date_max:%d/%m/%Y}.')
display(historique.groupby(['annee', 'code_type_local']).size().unstack(fill_value=0)
        .rename(columns={2: 'Appartements', 4: 'Locaux'}))
display(historique[['date_mutation', 'valeur_fonciere', 'surface_reelle', 'prix_m2']].head(5))

### 1.5. Évolution du prix au m² des appartements dans Paris

Le tableau conserve le nombre de transactions, la valeur foncière moyenne,
la surface moyenne, ainsi que la moyenne et la médiane du prix au m².
La médiane complète la moyenne pour apprécier la sensibilité aux valeurs extrêmes.

In [ ]:
appartements = historique.loc[historique['code_type_local'].eq(2)].copy()
locaux = historique.loc[historique['code_type_local'].eq(4)].copy()
annuel_appartements = appartements.groupby('annee').agg(
    transactions=('prix_m2', 'size'),
    valeur_moyenne_euros=('valeur_fonciere', 'mean'),
    surface_moyenne_m2=('surface_reelle', 'mean'),
    prix_m2_moyen=('prix_m2', 'mean'),
    prix_m2_median=('prix_m2', 'median'),
)
display(annuel_appartements)
variation_2017_2021 = 100 * (annuel_appartements.loc[2021, 'prix_m2_moyen'] / annuel_appartements.loc[2017, 'prix_m2_moyen'] - 1)
variation_2020_2021 = 100 * (annuel_appartements.loc[2021, 'prix_m2_moyen'] / annuel_appartements.loc[2020, 'prix_m2_moyen'] - 1)

fig, ax = plt.subplots()
ax.plot(annuel_appartements.index, annuel_appartements['prix_m2_moyen'], marker='o', color=BLEU, label='Moyenne')
ax.plot(annuel_appartements.index, annuel_appartements['prix_m2_median'], marker='s', linestyle='--', color=ORANGE, label='Médiane')
ax.set(title='Appartements à Paris — évolution du prix au m²', xlabel='Année de vente', ylabel='Prix (€/m²)', xticks=range(2017, 2022))
ax.yaxis.set_major_formatter(format_euros)
ax.legend(frameon=False)
terminer_figure(fig, '01_evolution_appartements')
display(Markdown(
    f'La moyenne passe de **{fr(annuel_appartements.loc[2017, "prix_m2_moyen"])} €/m² en 2017** '
    f'à **{fr(annuel_appartements.loc[2021, "prix_m2_moyen"])} €/m² en 2021**, '
    f'soit **{fr(variation_2017_2021, 1)} %**. Elle progresse jusqu’en 2020, '
    f'puis varie de **{fr(variation_2020_2021, 1)} % en 2021**. '
    'Ces moyennes ne corrigent ni l’inflation ni la composition des biens vendus ; '
    'elles ne mesurent pas l’évolution du prix d’un même appartement.'
))

### 1.6. Différences entre arrondissements

Je vérifie la présence des 20 codes postaux, puis j’affiche le premier arrondissement
et les 20 séries. Pour rendre les courbes lisibles, les arrondissements sont répartis
sur quatre panneaux de cinq séries, avec **la même échelle verticale**.

In [ ]:
codes_attendus = [str(cp) for cp in range(75001, 75021)]
codes_observes = sorted(historique['code_postal'].unique())
print('Codes postaux présents :', ', '.join(codes_observes))
assert codes_observes == codes_attendus
prix_arrondissements = appartements.pivot_table(
    index='code_postal', columns='annee', values='prix_m2', aggfunc='mean'
).reindex(index=codes_attendus, columns=range(2017, 2022))
effectifs_arrondissements = pd.crosstab(appartements['code_postal'], appartements['annee']).reindex(
    index=codes_attendus, columns=range(2017, 2022), fill_value=0
)
assert effectifs_arrondissements.to_numpy().sum() == len(appartements)
display(prix_arrondissements.round(0).astype('Int64').rename_axis('Code postal — moyenne €/m²'))
print('Effectif minimum par arrondissement et année :', int(effectifs_arrondissements.min().min()))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(prix_arrondissements.columns, prix_arrondissements.loc['75001'], marker='o', color=BLEU)
ax.set(title='1er arrondissement — prix moyen au m² des appartements', xlabel='Année de vente', ylabel='Prix (€/m²)', xticks=range(2017, 2022))
ax.yaxis.set_major_formatter(format_euros)
terminer_figure(fig, '02_premier_arrondissement')

fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharex=True, sharey=True)
palette = [BLEU, ORANGE, '#718044', '#9A5272', GRIS]
marqueurs = ['o', 's', '^', 'D', 'v']
for panneau, ax in enumerate(axes.flat):
    for i, cp in enumerate(codes_attendus[panneau * 5:(panneau + 1) * 5]):
        ax.plot(prix_arrondissements.columns, prix_arrondissements.loc[cp],
                color=palette[i], marker=marqueurs[i], label=cp, linewidth=1.5, markersize=4)
    ax.set(xlabel='Année de vente', ylabel='Prix (€/m²)', xticks=range(2017, 2022))
    ax.yaxis.set_major_formatter(format_euros)
    ax.legend(frameon=False, ncol=3, fontsize=9, loc='upper left')
fig.suptitle('Appartements — évolution dans les 20 arrondissements (2017–2021)')
terminer_figure(fig, '03_vingt_arrondissements')

In [ ]:
classement_2021 = prix_arrondissements[2021].sort_values(ascending=False)
cp_haut, cp_bas = classement_2021.index[0], classement_2021.index[-1]
display(Markdown(
    f'En 2021, la moyenne est la plus élevée dans le **{int(cp_haut[-2:])}ᵉ arrondissement** '
    f'(**{fr(classement_2021.iloc[0])} €/m²**) et la plus basse dans le '
    f'**{int(cp_bas[-2:])}ᵉ arrondissement** (**{fr(classement_2021.iloc[-1])} €/m²**). '
    'La localisation est donc une variable à examiner dans le modèle. '
    'Ces écarts restent descriptifs : les biens vendus ne présentent pas tous les mêmes caractéristiques.'
))

### 1.7. Prix par transaction dans le 6ᵉ arrondissement

In [ ]:
appartements_6e = appartements.loc[appartements['code_postal'].eq('75006')].sort_values('date_mutation').copy()
print(f'Appartements du 6ᵉ arrondissement : {fr(len(appartements_6e))} transactions représentées.')
display(appartements_6e.groupby('annee').agg(
    transactions=('prix_m2', 'size'), moyenne_euros_m2=('prix_m2', 'mean'),
    minimum_euros_m2=('prix_m2', 'min'), maximum_euros_m2=('prix_m2', 'max'),
))
fig, ax = plt.subplots()
ax.scatter(appartements_6e['date_mutation'], appartements_6e['prix_m2'], s=18, alpha=.45, color=BLEU)
ax.set(title='6ᵉ arrondissement — prix au m² de chaque appartement vendu', xlabel='Date de vente (2017–2021)', ylabel='Prix (€/m²)')
ax.set_xlim(pd.Timestamp('2017-01-01'), pd.Timestamp('2022-01-01'))
ax.set_xticks([pd.Timestamp(f'{an}-01-01') for an in range(2017, 2022)])
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.yaxis.set_major_formatter(format_euros)
terminer_figure(fig, '04_transactions_sixieme')
display(Markdown(
    f'Le prix au m² s’étend de **{fr(appartements_6e.prix_m2.min())}** à '
    f'**{fr(appartements_6e.prix_m2.max())} €/m²**. '
    'Le nuage montre une progression au début de la période puis un tassement. '
    'La dispersion entre ventes persiste : la date seule ne suffit pas à expliquer tous les prix.'
))

### 1.8. Corrélation entre le prix au m² et la date dans le 6ᵉ arrondissement

Le calcul demandé est celui de **Pearson**, sur les appartements du 6ᵉ uniquement.
La date devient le nombre d’années écoulées depuis le 01/01/2017. Le coefficient
mesure une association linéaire ; il ne prouve pas une causalité. Spearman est
présenté comme contrôle complémentaire de l’association monotone.

Hypothèse nulle : corrélation linéaire nulle ; seuil descriptif retenu : 5 %.
Les ventes peuvent partager des facteurs temporels et géographiques : les hypothèses
d’indépendance et de distribution du test ne sont pas garanties. La p-value reste indicative.

In [ ]:
r_date, p_date = pearsonr(appartements_6e['date_annees'], appartements_6e['prix_m2'])
rho_date, p_spearman = spearmanr(appartements_6e['date_annees'], appartements_6e['prix_m2'])
display(pd.DataFrame({
    'Méthode': ['Pearson (linéaire)', 'Spearman (monotone)'],
    'Coefficient': [r_date, rho_date],
    'p-value': [afficher_p(p_date), afficher_p(p_spearman)],
    'Nombre de ventes': [len(appartements_6e)] * 2,
}))
display(Markdown(
    f'**Pearson r = {fr(r_date, 3)}** : l’association observée entre la date et le prix au m² '
    'est positive et forte dans ce sous-ensemble. '
    f'La p-value ({afficher_p(p_date)}) est inférieure à 0,05 ; '
    'le test fournit un signal statistique d’association sous ses hypothèses. '
    'Cela ne démontre ni un effet causal du temps ni une hausse constante après 2021.'
))

### 1.9. Relation entre la valeur foncière et la surface

La consigne porte ici sur le **prix total**, et non sur le prix au m². Je calcule
la corrélation pour l’ensemble du fichier et pour chaque type de bien. Un contrôle
du prix au m² en fonction de la surface évite de confondre les deux mesures.

In [ ]:
resultats_correlations = []
for nom, sous_ensemble in [('Tous les biens', historique), ('Appartements', appartements), ('Locaux', locaux)]:
    for cible, nom_cible in [('valeur_fonciere', 'Valeur foncière (€)'), ('prix_m2', 'Prix au m² (€/m²)')]:
        r, p = pearsonr(sous_ensemble['surface_reelle'], sous_ensemble[cible])
        resultats_correlations.append({
            'Population': nom, 'Variable comparée à la surface': nom_cible,
            'n': len(sous_ensemble), 'Pearson r': r, 'p-value': afficher_p(p),
        })
table_correlations = pd.DataFrame(resultats_correlations)
display(table_correlations)
r_surface_valeur, p_surface_valeur = pearsonr(historique['surface_reelle'], historique['valeur_fonciere'])
r_surface_m2, p_surface_m2 = pearsonr(historique['surface_reelle'], historique['prix_m2'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for donnees, couleur, label, marker in [(appartements, BLEU, 'Appartement', 'o'), (locaux, ORANGE, 'Local', '^')]:
    axes[0].scatter(donnees['surface_reelle'], donnees['valeur_fonciere'], s=10, alpha=.25, color=couleur, label=label, marker=marker)
    axes[1].scatter(donnees['surface_reelle'], donnees['prix_m2'], s=10, alpha=.25, color=couleur, marker=marker)
axes[0].set(title='Surface et valeur foncière totale', xlabel='Surface réelle (m², échelle logarithmique)', ylabel='Valeur (€ ; échelle logarithmique)', xscale='log', yscale='log')
axes[0].legend(frameon=False)
axes[1].set(title='Surface et prix au m²', xlabel='Surface réelle (m², échelle logarithmique)', ylabel='Prix (€/m²)', xscale='log')
axes[1].yaxis.set_major_formatter(format_euros)
terminer_figure(fig, '05_surface_et_prix')
display(Markdown(
    f'La corrélation surface / valeur totale est **r = {fr(r_surface_valeur, 3)}** '
    f'(p {afficher_p(p_surface_valeur)}), contre **r = {fr(r_surface_m2, 3)}** '
    'pour surface / prix au m². La relation forte avec la valeur totale est cohérente '
    'avec l’identité « valeur = surface × prix au m² ». Elle ne signifie pas que les '
    'grandes surfaces ont un prix au m² proportionnellement plus élevé. '
    'Les axes logarithmiques rendent toutes les tailles visibles ; Pearson est calculé '
    'sur les valeurs originales, sans logarithme ni suppression des grands biens.'
))

### 1.10. Analyse des locaux et comparaison des deux types de biens

In [ ]:
annuel_locaux = locaux.groupby('annee').agg(
    transactions=('prix_m2', 'size'), surface_moyenne_m2=('surface_reelle', 'mean'),
    prix_m2_moyen=('prix_m2', 'mean'), prix_m2_median=('prix_m2', 'median'),
)
display(annuel_locaux)
comparaison_types = historique.groupby('type_local').agg(
    transactions=('prix_m2', 'size'), prix_m2_moyen=('prix_m2', 'mean'),
    prix_m2_median=('prix_m2', 'median'),
)
display(comparaison_types)
fig, ax = plt.subplots()
ax.plot(annuel_appartements.index, annuel_appartements['prix_m2_moyen'], marker='o', color=BLEU, label='Appartements')
ax.plot(annuel_locaux.index, annuel_locaux['prix_m2_moyen'], marker='s', linestyle='--', color=ORANGE, label='Locaux industriels, commerciaux ou assimilés')
ax.set(title='Paris — prix moyen au m² selon le type de bien', xlabel='Année de vente', ylabel='Prix (€/m²)', xticks=range(2017, 2022))
ax.yaxis.set_major_formatter(format_euros)
ax.legend(frameon=False, fontsize=10)
terminer_figure(fig, '06_comparaison_types')
ecart_types = 100 * (locaux['prix_m2'].mean() / appartements['prix_m2'].mean() - 1)
display(Markdown(
    f'Sur l’ensemble de 2017–2021, les locaux ont un prix moyen au m² '
    f'**{fr(ecart_types, 1)} % plus élevé** que celui des appartements dans ce fichier. '
    'Les niveaux moyens diffèrent, mais cela ne permet pas d’attribuer avec certitude '
    'un type à un bien individuel. La localisation et la période doivent aussi être prises en compte.'
))

### Bilan de l’analyse et choix des variables

| Variable | Rôle dans la régression | Justification et limite |
|---|---|---|
| Surface réelle (m²) | Numérique | Relation forte avec le prix total ; grands biens influents |
| Date, en années depuis 2017 | Numérique | Tendance temporelle ; une droite ne reproduit pas un retournement |
| Code postal | Catégorielle | Écarts entre arrondissements ; ne décrit pas le quartier ou l’immeuble |
| Type de bien | Catégorielle | Deux catégories de niveaux de prix différents |
| Valeur foncière (€) | **Cible** | Montant total à estimer |

Le **prix au m² observé est exclu des variables explicatives** : il est calculé
à partir de la cible et créerait une fuite d’information. Les adresses et noms de communes
sont conservés pour l’identification, mais ne sont pas utilisés dans ce premier modèle.
L’absence d’état du bien, d’étage ou de DPE limite la portée des estimations.

<a id="milestone-2"></a>
## Milestone 2 — Entraînement de l’algorithme

### 2.1. Définir la cible et réserver 33 % des observations

La cible est directement la **valeur foncière en euros**, conformément au sujet.
Les répétitions exactes sont retirées uniquement du jeu de modélisation : cela évite
la présence d’une observation identique dans les deux ensembles. Les données descriptives
de la première partie conservent toutes les lignes.

`train_test_split(test_size=0.33, random_state=42)` réserve 33 % des observations,
à l’arrondi près. La stratification préserve la proportion des deux types.
Le test reste à l’écart du calcul des coefficients et de l’apprentissage de l’encodage.

In [ ]:
base_modele = historique.loc[~historique_brut.duplicated()].copy()
VARIABLES_NUMERIQUES = ['surface_reelle', 'date_annees']
VARIABLES_CATEGORIELLES = ['code_postal', 'type_local']
VARIABLES = VARIABLES_NUMERIQUES + VARIABLES_CATEGORIELLES
X = base_modele[VARIABLES]
y = base_modele['valeur_fonciere']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=PART_TEST, random_state=GRAINE,
    stratify=base_modele['code_type_local'],
)
assert set(X_train.index).isdisjoint(X_test.index)
assert len(X_train) + len(X_test) == len(base_modele)
display(pd.DataFrame({
    'Ensemble': ['Base sans répétitions exactes', 'Apprentissage', 'Test indépendant'],
    'Lignes': [len(base_modele), len(X_train), len(X_test)],
    'Part de la base (%)': [100, 100 * len(X_train) / len(X), 100 * len(X_test) / len(X)],
}))

### 2.2. Encoder les catégories et entraîner la régression

Un **pipeline** regroupe l’encodage et la régression. L’encodeur est ajusté uniquement
sur l’apprentissage, puis réutilisé à l’identique pour le test et le portefeuille.
Une modalité de référence est retirée pour éviter la redondance avec l’interception.
Les variables numériques restent numériques : il ne faut pas convertir toutes les colonnes en texte.

In [ ]:
preparation = ColumnTransformer([
    ('numerique', 'passthrough', VARIABLES_NUMERIQUES),
    ('categoriel', OneHotEncoder(drop='first', handle_unknown='error'), VARIABLES_CATEGORIELLES),
], sparse_threshold=0)
modele_definition = Pipeline([
    ('preparation', preparation),
    ('regression', LinearRegression()),
])
modele_test = clone(modele_definition)
modele_test.fit(X_train, y_train)
prediction_test = modele_test.predict(X_test)
assert np.isfinite(prediction_test).all()
print('Nombre de variables après encodage :', modele_test['regression'].n_features_in_)
encodeur = modele_test['preparation'].named_transformers_['categoriel']
print('Catégories de référence :', {
    nom: categories[indice]
    for nom, categories, indice in zip(VARIABLES_CATEGORIELLES, encodeur.categories_, encodeur.drop_idx_)
})
print('Prédictions nulles ou négatives :', int((prediction_test <= 0).sum()))

### 2.3. Mesurer l’erreur de prédiction

- **MAPE (%)** : moyenne de `abs(prédiction − valeur réelle) / valeur réelle × 100`.
  C’est l’erreur moyenne en pourcentage demandée ; elle ne vaut pas `100 − R²`.
- **MAE (€)** : erreur absolue moyenne, dans l’unité du prix.
- **RMSE (€)** : racine de l’erreur quadratique moyenne ; pénalise davantage les grandes erreurs.
- **R²** : part de variance expliquée par rapport à une référence constante sur l’échantillon évalué.
- **Biais relatif moyen (%)** : une valeur positive indique une surestimation moyenne en pourcentage.

La MAPE donne le même poids à chaque bien et peut être sensible aux prix faibles.
Les prix sont tous strictement positifs. Un modèle naïf prédit la **médiane de l’apprentissage**
pour tous les biens ; il sert de point de comparaison simple.

In [ ]:
def mesurer_regression(reel, predit):
    reel, predit = np.asarray(reel), np.asarray(predit)
    return {
        'MAPE (%)': 100 * mean_absolute_percentage_error(reel, predit),
        'MAE (€)': mean_absolute_error(reel, predit),
        'RMSE (€)': np.sqrt(mean_squared_error(reel, predit)),
        'R²': r2_score(reel, predit),
        'Biais relatif (%)': 100 * np.mean((predit - reel) / reel),
    }

reference_mediane = DummyRegressor(strategy='median').fit(X_train, y_train)
metriques_test = mesurer_regression(y_test, prediction_test)
metriques_naives = mesurer_regression(y_test, reference_mediane.predict(X_test))
table_metriques = pd.DataFrame([metriques_naives, metriques_test],
    index=['Référence : médiane', 'Régression linéaire'])
display(table_metriques)

diagnostic_test = X_test.copy()
diagnostic_test['valeur_reelle'] = y_test
diagnostic_test['valeur_predite'] = prediction_test
diagnostic_test['erreur_absolue_euros'] = np.abs(prediction_test - y_test)
diagnostic_test['erreur_absolue_pct'] = 100 * diagnostic_test['erreur_absolue_euros'] / y_test
display(diagnostic_test.groupby('type_local').agg(
    biens=('valeur_reelle', 'size'), MAE_euros=('erreur_absolue_euros', 'mean'),
    MAPE_pct=('erreur_absolue_pct', 'mean'),
))
display(Markdown(
    f'La régression commet **{fr(metriques_test["MAPE (%)"], 2)} % d’erreur absolue relative moyenne** '
    f'sur le test de 33 %, soit une MAE de **{fr(metriques_test["MAE (€)"])} €**. '
    f'Le R² vaut **{fr(metriques_test["R²"], 3)}**. '
    'Ce sont des résultats hors apprentissage, mais le découpage aléatoire mélange les années : '
    'il ne suffit pas à établir la qualité d’une prévision future.'
))

### 2.4. Examiner les prédictions et les erreurs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for type_bien, couleur, label, marker in [(LIBELLES_TYPES[2], BLEU, 'Appartement', 'o'), (LIBELLES_TYPES[4], ORANGE, 'Local', '^')]:
    part = diagnostic_test.loc[diagnostic_test['type_local'].eq(type_bien)]
    axes[0].scatter(part['valeur_reelle'], part['valeur_predite'], s=12, alpha=.35, color=couleur, label=label, marker=marker)
minimum = min(y_test.min(), prediction_test.min())
maximum = max(y_test.max(), prediction_test.max())
axes[0].plot([minimum, maximum], [minimum, maximum], '--', color=GRIS, label='Prédiction exacte')
axes[0].set(title='Test de 33 % — valeurs réelles et prédites', xlabel='Valeur réelle (€ ; logarithmique)', ylabel='Valeur prédite (€ ; logarithmique)', xscale='log', yscale='log')
axes[0].legend(frameon=False, fontsize=9)
axes[1].hist(diagnostic_test['erreur_absolue_pct'], bins=35, color=BLEU, edgecolor='white')
axes[1].axvline(metriques_test['MAPE (%)'], color=ORANGE, linestyle='--', label=f'Moyenne : {fr(metriques_test["MAPE (%)"], 1)} %')
axes[1].set(title='Distribution des erreurs relatives', xlabel='Erreur absolue relative (%)', ylabel='Nombre de biens')
axes[1].legend(frameon=False, fontsize=9)
terminer_figure(fig, '07_diagnostic_regression')
display(Markdown(
    f'La médiane des erreurs relatives est de **{fr(diagnostic_test.erreur_absolue_pct.median(), 2)} %** ; '
    f'90 % des erreurs sont inférieures ou égales à **{fr(diagnostic_test.erreur_absolue_pct.quantile(.9), 2)} %**. '
    'Ces quantiles décrivent le test observé ; ils ne constituent pas des intervalles de confiance pour le portefeuille.'
))

### 2.5. Complément : tester une prévision dans le temps

Pour se rapprocher de l’utilisation prévue en 2022, une seconde évaluation conserve
2017–2020 pour apprendre et **toute l’année 2021 pour tester**. Elle complète le test
de 33 % demandé. Le même modèle et les mêmes variables sont utilisés, sans ajustement
choisi à partir de ce second test.

In [ ]:
masque_avant_2021 = base_modele['date_mutation'].lt('2021-01-01')
modele_chronologique = clone(modele_definition)
modele_chronologique.fit(X.loc[masque_avant_2021], y.loc[masque_avant_2021])
prediction_2021 = modele_chronologique.predict(X.loc[~masque_avant_2021])
metriques_2021 = mesurer_regression(y.loc[~masque_avant_2021], prediction_2021)
display(pd.DataFrame([metriques_test, metriques_2021], index=['Test aléatoire 33 %', 'Test chronologique 2021']))
print('Apprentissage chronologique :', int(masque_avant_2021.sum()), 'biens ; test 2021 :', int((~masque_avant_2021).sum()), 'biens.')
display(Markdown(
    f'La MAPE atteint **{fr(metriques_2021["MAPE (%)"], 2)} % sur 2021**, '
    f'avec un biais relatif moyen de **{fr(metriques_2021["Biais relatif (%)"], 2)} %**. '
    'La prévision temporelle est plus difficile que l’interpolation entre ventes mélangées. '
    'La tendance linéaire ne reproduit pas complètement le tassement observé en 2021.'
))

### Conclusions sur la régression et pistes d’amélioration

1. La régression explique une grande partie de la valeur totale grâce à la surface,
   mais la moyenne d’erreur ne garantit pas la précision pour chaque bien.
2. Le contrôle chronologique est le plus proche de l’usage futur. Son biais doit être
   communiqué avec la valorisation 2022.
3. L’état, l’étage, le DPE, le quartier précis et la nature détaillée des locaux
   pourraient améliorer le modèle s’ils étaient disponibles de façon fiable.
4. Des interactions entre surface, type et localisation, une cible logarithmique ou
   une méthode non linéaire pourraient être comparées par validation chronologique.
   Ces variantes ne sont pas évaluées ici ; aucune amélioration de score n’est présumée.
5. Avant la valorisation finale, le pipeline sera réentraîné sur toutes les observations
   historiques distinctes de 2017–2021. Les scores ci-dessus restent ceux des tests tenus à l’écart.

<a id="milestone-3"></a>
## Milestone 3 — Prédiction pour le client

### 3.1. Importer et contrôler le portefeuille

La surface utilisée est **`surface_reelle_bati`**, renommée `surface_reelle` pour
le modèle. La surface Carrez ne lui est pas substituée. Le numéro `id_bien` créé
ci-dessous repère la ligne du classeur ; ce n’est pas un identifiant cadastral.
Les lignes entièrement identiques sont signalées et conservées : deux lignes
semblables pourraient représenter deux actifs différents.

In [ ]:
portefeuille_brut = pd.read_excel(DATA_DIR / FICHIERS['portefeuille'], engine='openpyxl')
verifier_colonnes(portefeuille_brut, [
    'surface_reelle_bati', 'code_postal', 'code_type_local', 'type_local',
    'adresse_numero', 'adresse_nom_voie',
], 'Portefeuille')
nb_identiques_portefeuille = int(portefeuille_brut.duplicated().sum())
portefeuille = portefeuille_brut.copy()
for colonne in ['surface_reelle_bati', 'code_postal', 'code_type_local']:
    portefeuille[colonne] = pd.to_numeric(portefeuille[colonne], errors='coerce')
colonnes_controle = ['surface_reelle_bati', 'code_postal', 'code_type_local']
if (not np.isfinite(portefeuille[colonnes_controle]).all().all()
        or portefeuille['surface_reelle_bati'].le(0).any()):
    raise ValueError('Le portefeuille contient des surfaces ou des codes invalides.')
if not portefeuille['code_type_local'].isin(LIBELLES_TYPES).all():
    raise ValueError('Un type de bien du portefeuille est inconnu.')
if not portefeuille['type_local'].eq(portefeuille['code_type_local'].map(LIBELLES_TYPES)).all():
    raise ValueError('Les codes et libellés du portefeuille ne correspondent pas.')
if not portefeuille['code_postal'].isin(range(75001, 75021)).all():
    raise ValueError('Un code postal du portefeuille est hors du périmètre parisien.')
portefeuille['code_postal'] = portefeuille['code_postal'].astype(int).astype(str)
portefeuille['code_type_local'] = portefeuille['code_type_local'].astype(int)
portefeuille['surface_reelle'] = portefeuille['surface_reelle_bati']
portefeuille['date_mutation'] = DATE_ESTIMATION
portefeuille['date_annees'] = (DATE_ESTIMATION - DATE_ORIGINE).days / 365.25
portefeuille.insert(0, 'id_bien', np.arange(1, len(portefeuille) + 1))
portefeuille['segment_conventionnel'] = portefeuille['code_type_local'].map(SEGMENTS)
print(f'{len(portefeuille)} biens ; {nb_identiques_portefeuille} répétitions exactes signalées et conservées.')
display(portefeuille[['id_bien', 'code_postal', 'type_local', 'surface_reelle', 'surface_carrez']].head())

### 3.2. Réentraîner le modèle final et estimer chaque actif

Le modèle final apprend sur l’historique **2017–2021 complet sans répétitions exactes**.
Le pipeline réutilise le même schéma pour le portefeuille. Les catégories nouvelles,
surfaces hors du domaine d’apprentissage et prédictions non positives sont contrôlées.
La date du 31/12/2022 dépasse l’historique d’un an : cette extrapolation est explicite.

In [ ]:
modele_final = clone(modele_definition)
modele_final.fit(X, y)
for colonne in VARIABLES_CATEGORIELLES:
    inconnues = set(portefeuille[colonne]) - set(X[colonne])
    if inconnues:
        raise ValueError(f'Catégories absentes de l’apprentissage pour {colonne} : {inconnues}')

portefeuille['surface_hors_plage_historique'] = ~portefeuille['surface_reelle'].between(
    base_modele['surface_reelle'].min(), base_modele['surface_reelle'].max()
)
portefeuille['valeur_estimee_euros'] = modele_final.predict(portefeuille[VARIABLES])
if (not np.isfinite(portefeuille['valeur_estimee_euros']).all()
        or portefeuille['valeur_estimee_euros'].le(0).any()):
    raise ValueError('Une estimation est non finie ou non positive : ne pas valider la valorisation.')
portefeuille['prix_m2_estime'] = portefeuille['valeur_estimee_euros'] / portefeuille['surface_reelle']
print('Actifs dont la surface est hors de la plage historique :', int(portefeuille['surface_hors_plage_historique'].sum()))
print('Écart entre la dernière vente observée et la date de prévision :', (DATE_ESTIMATION - date_max).days, 'jours.')
display(portefeuille[[
    'id_bien', 'code_postal', 'segment_conventionnel', 'surface_reelle',
    'prix_m2_estime', 'valeur_estimee_euros',
]].head(10))

### 3.3. Comparer les valorisations des segments

La ventilation suit la convention **appartements = particuliers** et **locaux = corporate**.
Elle doit être validée avec l’entreprise puisque le fichier ne contient aucune colonne de branche.
Les totaux sont additionnés en euros avant conversion en millions ; les valeurs ne sont pas
arrondies avant l’agrégation.

In [ ]:
valorisation_segments = portefeuille.groupby('segment_conventionnel').agg(
    nombre_biens=('id_bien', 'size'), surface_totale_m2=('surface_reelle', 'sum'),
    valeur_estimee_euros=('valeur_estimee_euros', 'sum'),
    valeur_moyenne_par_bien=('valeur_estimee_euros', 'mean'),
).reindex(['Particuliers', 'Corporate'])
valorisation_segments['valorisation_MEUR'] = valorisation_segments['valeur_estimee_euros'] / 1_000_000
total_portefeuille = portefeuille['valeur_estimee_euros'].sum()
valorisation_segments['part_valorisation_pct'] = 100 * valorisation_segments['valeur_estimee_euros'] / total_portefeuille
assert valorisation_segments['nombre_biens'].sum() == len(portefeuille)
assert np.isclose(valorisation_segments['valeur_estimee_euros'].sum(), total_portefeuille)
display(valorisation_segments)

fig, ax = plt.subplots(figsize=(9, 4.4))
barres = ax.barh(valorisation_segments.index, valorisation_segments['valorisation_MEUR'], color=[BLEU, ORANGE])
ax.bar_label(barres, labels=[f'{fr(v, 2)} M€' for v in valorisation_segments['valorisation_MEUR']], padding=6)
ax.set_xlim(0, valorisation_segments['valorisation_MEUR'].max() * 1.22)
ax.set(title='Portefeuille — valorisation estimée au 31/12/2022', xlabel='Millions d’euros — segments définis par convention')
ax.grid(axis='y', visible=False)
terminer_figure(fig, '08_valorisation_portefeuille')

segment_dominant = valorisation_segments['valeur_estimee_euros'].idxmax()
display(Markdown(
    f'**Particuliers : {fr(valorisation_segments.loc["Particuliers", "valorisation_MEUR"], 2)} M€** ; '
    f'**corporate : {fr(valorisation_segments.loc["Corporate", "valorisation_MEUR"], 2)} M€** ; '
    f'**total : {fr(total_portefeuille / 1_000_000, 2)} M€**. '
    f'Sous cette convention, le segment **{segment_dominant.lower()}** présente la valorisation totale '
    'la plus élevée. Ce résultat dépend du nombre d’actifs, de leur surface et de leurs caractéristiques ; '
    'il ne mesure pas la rentabilité, la croissance ni l’intérêt d’un arbitrage.'
))

### Limites de la valorisation

- La prévision au 31/12/2022 extrapole une tendance apprise jusqu’en 2021 ;
  le contrôle chronologique montre déjà une dégradation et un biais de surestimation.
- Les scores du test ne sont pas une mesure d’erreur sur ces actifs : leurs prix réels
  au 31/12/2022 ne sont pas fournis. Aucun intervalle de confiance du total n’est estimé.
- Les lignes identiques du portefeuille doivent être vérifiées auprès du propriétaire
  avant toute utilisation opérationnelle des totaux.
- La segmentation doit être confirmée avec l’entreprise. Une expertise individuelle
  et des données plus récentes seraient nécessaires pour une décision d’achat ou de vente.

<a id="milestone-4"></a>
## Milestone 4 — Classification des données de Louise

### 4.1. Importer et préparer l’échantillon à classer

K-means est une méthode **non supervisée** : elle sépare les observations en groupes,
sans connaître leur type. Conformément au sujet, elle est ajustée ici sur **l’échantillon
fourni**, avec `n_clusters=2` et le seul **prix au m²** comme variable.

Le prix total et la surface servent à calculer ce ratio, puis sont exclus des variables
du clustering. Le code postal et le nom de commune sont constants dans ce fichier.
La date, absente, n’est pas inventée ni utilisée dans le modèle.

In [ ]:
echantillon_brut = pd.read_excel(DATA_DIR / FICHIERS['echantillon'], engine='openpyxl')
verifier_colonnes(echantillon_brut, ['valeur_fonciere', 'surface_reelle', 'code_postal'], 'Échantillon')
echantillon = echantillon_brut.copy()
for colonne in ['valeur_fonciere', 'surface_reelle', 'code_postal']:
    echantillon[colonne] = pd.to_numeric(echantillon[colonne], errors='coerce')
if not np.isfinite(echantillon[['valeur_fonciere', 'surface_reelle', 'code_postal']]).all().all():
    raise ValueError('L’échantillon contient des valeurs manquantes ou non finies.')
if echantillon[['valeur_fonciere', 'surface_reelle']].le(0).any().any():
    raise ValueError('Les prix et les surfaces de l’échantillon doivent être positifs.')
echantillon['code_postal'] = echantillon['code_postal'].astype(int).astype(str)
if echantillon['code_postal'].nunique() != 1 or echantillon['code_postal'].iloc[0] != '75019':
    raise ValueError('Le périmètre a changé : réexaminer la comparaison historique avant de classer.')
echantillon.insert(0, 'id_opportunite', np.arange(1, len(echantillon) + 1))
echantillon['prix_m2'] = echantillon['valeur_fonciere'] / echantillon['surface_reelle']
X_kmeans = echantillon[['prix_m2']].copy()
print(f'{len(echantillon)} opportunités, toutes dans le 19ᵉ arrondissement.')
print('Colonne de date présente :', 'date_mutation' in echantillon_brut.columns)
print('Répétitions exactes :', int(echantillon_brut.duplicated().sum()))
display(echantillon[['id_opportunite', 'valeur_fonciere', 'surface_reelle', 'prix_m2']].head(10))

### 4.2. Ajuster K-means et examiner les deux centres

Avec une seule variable numérique, une standardisation affine ne change pas la
partition recherchée. Je conserve les euros par m² pour interpréter directement les centres.
`n_init=20` teste plusieurs initialisations ; la graine fixe rend le résultat reproductible.

Les numéros **0 et 1 n’ont aucun sens métier**. Les noms « prix au m² bas » et
« prix au m² haut » sont déterminés par les centres calculés, sans supposer qu’un numéro
correspond toujours à un type de bien.

In [ ]:
kmeans = KMeans(n_clusters=2, n_init=20, random_state=GRAINE)
echantillon['cluster'] = kmeans.fit_predict(X_kmeans)
centres = pd.Series(kmeans.cluster_centers_.ravel(), index=range(2), name='Centre (€/m²)')
cluster_bas = int(centres.idxmin())
cluster_haut = int(centres.idxmax())
noms_groupes = {cluster_bas: 'Prix au m² bas', cluster_haut: 'Prix au m² haut'}
echantillon['groupe_prix'] = echantillon['cluster'].map(noms_groupes)
resume_clusters = echantillon.groupby('cluster').agg(
    effectif=('id_opportunite', 'size'), minimum_euros_m2=('prix_m2', 'min'),
    centre_euros_m2=('prix_m2', 'mean'), maximum_euros_m2=('prix_m2', 'max'),
)
resume_clusters['groupe'] = resume_clusters.index.map(noms_groupes)
resume_clusters = resume_clusters.sort_values('centre_euros_m2')
score_silhouette = silhouette_score(X_kmeans, echantillon['cluster'])
seuil_kmeans = centres.mean()
display(resume_clusters)
print(f'Silhouette : {score_silhouette:.3f} ; frontière entre les deux centres : {fr(seuil_kmeans)} €/m².')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
for cluster, couleur, marqueur in [(cluster_bas, BLEU, 'o'), (cluster_haut, ORANGE, '^')]:
    part = echantillon.loc[echantillon['cluster'].eq(cluster)]
    ax.scatter(part['id_opportunite'], part['prix_m2'], color=couleur, marker=marqueur,
               s=40, label=f'{noms_groupes[cluster]} (n = {len(part)})')
    ax.axhline(centres.loc[cluster], color=couleur, linestyle=':', linewidth=1.5)
ax.axhline(seuil_kmeans, color=GRIS, linestyle='--', label='Frontière K-means')
ax.set(title='Louise — deux groupes de prix au m² dans le 19ᵉ arrondissement',
       xlabel='Identifiant de ligne dans l’échantillon', ylabel='Prix (€/m²)')
ax.yaxis.set_major_formatter(format_euros)
ax.legend(frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=3, fontsize=9)
terminer_figure(fig, '09_kmeans_echantillon')
display(Markdown(
    f'Les deux groupes contiennent **{int(resume_clusters.iloc[0].effectif)}** et '
    f'**{int(resume_clusters.iloc[1].effectif)} biens**, avec des centres à '
    f'**{fr(centres.min())} €/m²** et **{fr(centres.max())} €/m²**. '
    f'La silhouette de **{fr(score_silhouette, 3)}** indique une séparation nette des prix. '
    'Elle ne mesure pas le pourcentage de types de biens correctement identifiés.'
))

### 4.3. Donner un sens aux groupes avec prudence

Pour proposer les deux libellés demandés, je compare les groupes aux transactions
**du même arrondissement en 2021**. Ce repère est préférable à une moyenne sur tout Paris,
où les écarts entre arrondissements pourraient masquer les écarts entre types.
Il sert à interpréter le résultat ; il ne transforme pas les types supposés en vérité terrain.

In [ ]:
reference_19e = historique.loc[
    historique['code_postal'].eq('75019') & historique['annee'].eq(ANNEE_ECHANTILLON)
]
resume_reference_19e = reference_19e.groupby('code_type_local').agg(
    effectif=('prix_m2', 'size'), moyenne_euros_m2=('prix_m2', 'mean'),
    mediane_euros_m2=('prix_m2', 'median'), minimum_euros_m2=('prix_m2', 'min'),
    maximum_euros_m2=('prix_m2', 'max'),
)
display(resume_reference_19e.rename(index=LIBELLES_TYPES))
ordre_types = resume_reference_19e['moyenne_euros_m2'].sort_values().index.tolist()
if set(ordre_types) != {2, 4}:
    raise ValueError('Les deux types ne sont pas représentés dans la référence locale.')
correspondance_types = {cluster_bas: ordre_types[0], cluster_haut: ordre_types[1]}
echantillon['code_type_estime'] = echantillon['cluster'].map(correspondance_types)
echantillon['type_estime'] = echantillon['code_type_estime'].map(LIBELLES_TYPES)
echantillon['statut'] = 'Hypothèse à confirmer'

centre_appartement = centres.loc[[c for c, t in correspondance_types.items() if t == 2][0]]
ecart_centre_reference = 100 * (centre_appartement / resume_reference_19e.loc[2, 'moyenne_euros_m2'] - 1)
display(Markdown(
    'Les locaux ont un prix moyen au m² plus élevé que les appartements dans la référence locale. '
    'Je propose donc **groupe bas → appartement** et **groupe haut → local**, sous réserve de validation. '
    f'Le centre du groupe supposé « appartement » reste toutefois **{fr(abs(ecart_centre_reference), 1)} % '
    'en dessous** de la moyenne des appartements du 19ᵉ en 2021. '
    'Ce décalage doit être éclairci : période différente, caractéristiques particulières ou contenu '
    'de l’échantillon peuvent être en cause. La date étant absente, aucune de ces explications '
    'n’est vérifiée. La référence locale ne contient en outre que '
    f'**{int(resume_reference_19e.loc[4, "effectif"])} locaux**.'
))

### 4.4. Afficher le classement provisoire des 40 biens

In [ ]:
display(echantillon['type_estime'].value_counts().rename_axis('Type proposé').to_frame('Nombre de biens'))
with pd.option_context('display.max_rows', len(echantillon)):
    display(echantillon[[
        'id_opportunite', 'code_postal', 'surface_reelle', 'valeur_fonciere',
        'prix_m2', 'groupe_prix', 'type_estime',
    ]].sort_values('id_opportunite'))

### 4.5. Validation métier : ce qui peut et ne peut pas être calculé

Le fichier ne fournit pas le type réel des opportunités. Il est donc impossible
de calculer une exactitude, une précision, un rappel ou une matrice de confusion
sur ces 40 biens à ce stade. Le score de silhouette est disponible, mais ne remplace pas ces mesures.

Si l’équipe de Louise fournit les types réels, les enregistrer dans
`data/raw/labels_reference_louise.csv` avec deux colonnes :
`id_opportunite` (numéro de ligne présenté ci-dessus) et `code_type_reel` (2 ou 4).
La cellule suivante vérifiera l’appariement complet et calculera les métriques.
Les labels réels ne sont jamais utilisés pour ajuster K-means ou changer sa correspondance.

In [ ]:
fichier_labels = DATA_DIR / 'labels_reference_louise.csv'
validation_louise = None
if not fichier_labels.exists():
    display(Markdown(
        '**Validation externe non disponible :** les types réels de Louise ne sont pas fournis. '
        'Le classement reste provisoire ; aucune performance de classification n’est annoncée.'
    ))
else:
    labels = pd.read_csv(fichier_labels, sep=None, engine='python')
    verifier_colonnes(labels, ['id_opportunite', 'code_type_reel'], 'Labels Louise')
    for colonne in ['id_opportunite', 'code_type_reel']:
        labels[colonne] = pd.to_numeric(labels[colonne], errors='raise')
    if labels['id_opportunite'].duplicated().any():
        raise ValueError('Plusieurs types réels sont associés à une même opportunité.')
    if set(labels['id_opportunite']) != set(echantillon['id_opportunite']):
        raise ValueError('Les labels doivent couvrir exactement les identifiants de l’échantillon.')
    if not labels['code_type_reel'].isin([2, 4]).all():
        raise ValueError('Les codes réels doivent être 2 (appartement) ou 4 (local).')
    controle_labels = echantillon.merge(labels, on='id_opportunite', validate='one_to_one')
    reel = controle_labels['code_type_reel']
    predit = controle_labels['code_type_estime']
    validation_louise = {
        'Exactitude': accuracy_score(reel, predit),
        'Exactitude équilibrée': balanced_accuracy_score(reel, predit),
        'Précision locaux': precision_score(reel, predit, pos_label=4, zero_division=0),
        'Rappel locaux': recall_score(reel, predit, pos_label=4, zero_division=0),
        'F1 locaux': f1_score(reel, predit, pos_label=4, zero_division=0),
    }
    display(pd.Series(validation_louise, name='Score entre 0 et 1').to_frame())
    display(pd.DataFrame(confusion_matrix(reel, predit, labels=[2, 4]),
        index=['Réel : appartement', 'Réel : local'],
        columns=['Prédit : appartement', 'Prédit : local']))

### Conclusion sur le classement

Le K-means demandé produit deux groupes de prix bien séparés. Le nom du type est une
**interprétation de ces groupes**, fondée sur l’ordre des moyennes historiques locales.
Le décalage entre le groupe bas et les appartements de référence fragilise cette hypothèse.
Je recommande d’obtenir les labels réels et les dates de l’échantillon avant de l’utiliser
pour trier automatiquement des opportunités. Avec des exemples étiquetés fiables,
une classification supervisée pourrait être évaluée sur un véritable jeu de test indépendant.

<a id="milestone-5"></a>
## Milestone 5 — Analyse et présentation des résultats

### 5.1. Synthèse des résultats calculés

In [ ]:
synthese = f"""
1. **Données :** {fr(len(historique))} lignes de ventes du {date_min:%d/%m/%Y} au {date_max:%d/%m/%Y},
   dont {fr(nb_appartements)} appartements et {fr(nb_locaux)} locaux.
2. **Prix des appartements :** {fr(variation_2017_2021, 1)} % entre les moyennes 2017 et 2021 ;
   {fr(variation_2020_2021, 1)} % de 2020 à 2021. Le prix moyen varie aussi selon l’arrondissement.
3. **Régression :** MAPE de {fr(metriques_test['MAPE (%)'], 2)} % sur le test de 33 %,
   et de {fr(metriques_2021['MAPE (%)'], 2)} % sur le test chronologique 2021.
   Ce dernier présente un biais relatif moyen de {fr(metriques_2021['Biais relatif (%)'], 2)} %.
4. **Portefeuille au 31/12/2022 :** {fr(total_portefeuille / 1_000_000, 2)} M€ estimés pour
   {len(portefeuille)} actifs, dont {fr(valorisation_segments.loc['Particuliers', 'valorisation_MEUR'], 2)} M€
   pour les particuliers et {fr(valorisation_segments.loc['Corporate', 'valorisation_MEUR'], 2)} M€
   pour le corporate, selon la convention fondée sur le type de bien. Le total corporate est supérieur.
5. **Louise :** {len(echantillon)} biens séparés en deux groupes de
   {int(resume_clusters.iloc[0].effectif)} et {int(resume_clusters.iloc[1].effectif)} observations.
   Silhouette : {fr(score_silhouette, 3)}. Les types proposés ne sont pas validés par des labels réels.
"""
display(Markdown(synthese))

### 5.2. Réponse au client et limites de l’exercice

Les données permettent de décrire l’évolution des prix, de produire une première
estimation du portefeuille et de construire un classement exploratoire.
Sous la convention de segmentation retenue, **le corporate possède la valorisation totale
estimée la plus élevée**. Cette comparaison ne permet pas de conclure que ce segment
est plus rentable ni de recommander sa vente ou son développement.

| Point à retenir | Conséquence pour l’utilisation |
|---|---|
| Historique pédagogique limité à 2017–2021 | Ne pas présenter les résultats comme les prix actuels du marché |
| Répétitions exactes et absence d’identifiants de mutation | Les décomptes descriptifs sont des nombres de lignes ; vérifier les actifs répétés |
| Corrélations et moyennes non corrigées de la composition | Ne pas transformer une association en explication causale |
| Erreur chronologique et biais de la régression | Communiquer une estimation indicative, sans précision individuelle garantie |
| Date 2022 hors historique | Mettre à jour les données et vérifier la stabilité du modèle avant usage réel |
| Deux segments déduits des types | Faire confirmer la correspondance organisationnelle par l’entreprise |
| Échantillon Louise sans date ni type réel | Valider les hypothèses et le classement avec l’équipe de Louise |

**Priorités proposées :** confirmer les doublons et les segments du portefeuille,
récupérer les labels et les dates des opportunités, puis enrichir les caractéristiques
des biens et réévaluer les modèles dans le temps. Le prix au m² seul ne suffit pas
à garantir le type réel d’un bien.

### 5.3. Trame de présentation orale

1. Présenter la question métier, les trois fichiers et la période couverte.
2. Montrer l’évolution annuelle, les écarts entre arrondissements et les deux types de biens.
3. Expliquer la cible, les quatre variables, le test de 33 % et la MAPE.
4. Comparer le test aléatoire au test chronologique, puis présenter les deux valorisations.
5. Montrer les deux groupes K-means et distinguer silhouette, type supposé et type réel.
6. Terminer par les limites et les informations à obtenir avant une utilisation opérationnelle.

Ce notebook fournit les éléments de l’analyse et de la présentation des résultats ;
il ne constitue pas un diaporama séparé.

### 5.4. Exporter les résultats et conserver la traçabilité

In [ ]:
colonnes_portefeuille = [
    'id_bien', 'adresse_numero', 'adresse_nom_voie', 'code_postal', 'type_local',
    'segment_conventionnel', 'surface_reelle', 'date_mutation', 'prix_m2_estime',
    'valeur_estimee_euros', 'surface_hors_plage_historique',
]
export_portefeuille = DOSSIER_RESULTATS / 'predictions_valorisation_portefeuille_2022-12-31.csv'
export_classification = DOSSIER_RESULTATS / 'classification_echantillon_louise_102026.csv'
portefeuille[colonnes_portefeuille].to_csv(export_portefeuille, index=False, sep=';', encoding='utf-8-sig', float_format='%.2f')
echantillon.to_csv(export_classification, index=False, sep=';', encoding='utf-8-sig', float_format='%.2f')
valorisation_segments.to_csv(DOSSIER_RESULTATS / 'synthese_valorisation_segments.csv', sep=';', encoding='utf-8-sig', float_format='%.2f')
pd.DataFrame([metriques_test, metriques_2021], index=['test_33_pct', 'test_2021']).to_csv(
    DOSSIER_RESULTATS / 'metriques_regression.csv', sep=';', encoding='utf-8-sig', float_format='%.6f')
annuel_appartements.to_csv(DOSSIER_RESULTATS / 'evolution_appartements.csv', sep=';', encoding='utf-8-sig', float_format='%.2f')

provenance = pd.DataFrame([
    {'Fichier': nom, 'SHA-256': sha256((DATA_DIR / nom).read_bytes()).hexdigest()}
    for nom in FICHIERS.values()
])
provenance.to_csv(DOSSIER_RESULTATS / 'sources_sha256.csv', index=False)
environnement = {'Python': platform.python_version()}
environnement.update({nom: version(nom) for nom in [
    'pandas', 'numpy', 'matplotlib', 'scipy', 'scikit-learn', 'openpyxl',
]})
(DOSSIER_RESULTATS / 'environnement.json').write_text(
    json.dumps(environnement, ensure_ascii=False, indent=2), encoding='utf-8'
)
display(pd.Series(environnement, name='Version utilisée').to_frame())
print('Exports créés dans livrables/ : valorisation, classification, synthèses, graphiques et traçabilité.')

### 5.5. Contrôles finaux et couverture des consignes

Ces vérifications rapprochent les totaux et les fichiers exportés des objets calculés.
La présence de résultats ne remplace pas la validation des hypothèses métier.

In [ ]:
verif_portefeuille = pd.read_csv(export_portefeuille, sep=';')
verif_classification = pd.read_csv(export_classification, sep=';')
assert len(verif_portefeuille) == len(portefeuille_brut)
assert verif_portefeuille['id_bien'].is_unique
assert np.isclose(verif_portefeuille['valeur_estimee_euros'].sum(), total_portefeuille, atol=0.01 * len(portefeuille), rtol=0)
assert len(verif_classification) == len(echantillon_brut)
assert verif_classification['id_opportunite'].is_unique
assert verif_classification['type_estime'].notna().all()
assert len(X_train.index.intersection(X_test.index)) == 0
assert np.allclose(historique['prix_m2'] * historique['surface_reelle'], historique['valeur_fonciere'])
assert np.isclose(np.mean(np.abs(prediction_test - y_test) / y_test) * 100, metriques_test['MAPE (%)'])
assert (appartements_6e['code_postal'] == '75006').all()
assert (appartements_6e['code_type_local'] == 2).all()
print('Contrôles de cohérence et rapprochement des exports : OK.')

| Demande du notebook fourni | Emplacement | État |
|---|---|---|
| Types, qualité, volumes et intervalle historique | 1.1 à 1.4 | Calculés et commentés |
| Prix au m², prix de vente moyen et surface moyenne par année | 1.5 | Présents |
| Premier arrondissement et ensemble des 20 arrondissements | 1.6 | Tableau et graphiques |
| Transactions et Pearson prix/date dans le 6ᵉ | 1.7 et 1.8 | Périmètre respecté |
| Corrélation valeur foncière/surface | 1.9 | Calculée ; prix au m² distingué |
| Analyse annuelle des locaux et comparaison aux appartements | 1.10 | Tableau et graphique |
| Encodage, régression linéaire et test de 33 % | 2.1 à 2.3 | Présents, sans fuite de la cible |
| Erreur moyenne en pourcentage et conclusions | 2.3 à 2.5 | MAPE et métriques complémentaires |
| Portefeuille au 31/12/2022 avec la surface réelle | 3.1 et 3.2 | Calculé et exporté |
| Comparaison particuliers/corporate et limites | 3.3 | Présente, convention explicitée |
| K-means à deux groupes sur l’échantillon transmis | 4.1 à 4.4 | Calculé ; labels déduits des centres |
| Validation des types réels des opportunités | 4.5 | En attente des labels non fournis ; code prêt |
| Conclusions, recommandations et présentation des résultats | 5.1 à 5.3 | Présentes |
| Exports, environnement et contrôles finaux | 5.4 et 5.5 | Présents |

### Références méthodologiques

- [SciPy — coefficient de Pearson](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.pearsonr.html)
- [scikit-learn — KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)
- [scikit-learn — erreur absolue relative moyenne](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_percentage_error.html)